In [3]:
!pip install pypdf


     ------------------------------------ 401.7/401.7 kB 511.1 kB/s eta 0:00:00



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import re
import json
from pypdf import PdfReader
from laya import Router

# ۱. لود صحیح مدل آموزش‌دیده محلی به صورت دیکشنری
model_dir = "./contract-risk-model"
print(f"Loading local checkpoint from: {model_dir}")

agent = Router(
    models={"contract_risk": model_dir},
    default="contract_risk"
)
print("✅ مدل محلی با موفقیت لود شد.\n")

# ۲. استخراج و تجمیع متن PDF
pdf_path = "1642490006-KttS.pdf"
reader = PdfReader(pdf_path)

full_text = ""
for page in reader.pages:
    txt = page.extract_text()
    if txt:
        full_text += "\n" + txt

# ۳. بخش‌بندی ساختاریافته (بر اساس تیتر ماده‌ها یا بندها)
# شناسایی عباراتی مثل "ماده یک", "ماده 1", "0-1", "4-2"
pattern = r'(?m)(?=^(?:ماده\s+[\u0600-\u06FF\d]+|\d+-\d+))'
raw_sections = re.split(pattern, full_text)

cleaned_clauses = []
for sec in raw_sections:
    cleaned = " ".join(sec.split()).strip()
    if len(cleaned) >= 30:
        cleaned_clauses.append(cleaned)

print(f"تعداد {len(cleaned_clauses)} بند کامل برای ارزیابی استخراج شد.\n")

# ۴. اسکیما
questions = {
    "risk_level": {
        "type": "choice",
        "instructions": "میزان ریسک حقوقی و مالی این بند برای فریلنسر چقدر است؟",
        "criteria": {
            "مناسب": "تعهدات شفاف، پرداخت به موقع، شرایط فسخ عادلانه و دوطرفه",
            "کم‌ریسک": "نیاز به شفاف‌سازی جزئی یا ابهام ملایم در زمان‌بندی تحویل",
            "پرریسک": "یک‌طرفه بودن، جریمه‌های نامتعارف، سلب مالکیت معنوی مطلق یا فسخ بدون دلیل"
        }
    }
}

# ۵. ارزیابی بند به بند و ثبت در لیست دیکشنری
analysis_results = []

for idx, clause in enumerate(cleaned_clauses, 1):
    try:
        res = agent.predict(clause, questions)
        
        ans = res.get("answers", {}).get("risk_level") or res.get("answers", {}).get("label", {})
        usage = res.get("usage", {})
        
        item = {
            "id": idx,
            "clause_text": clause,             # متن کامل بند
            "char_count": len(clause),          # تعداد کاراکتر متن ارسالی
            "predicted_risk": ans.get("choice", "نامشخص"),
            "confidence": ans.get("confidence"),
            "probabilities": ans.get("probabilities", {}),
            "tokens_used": usage.get("input_tokens"),
            "was_truncated": usage.get("truncated", False)  # بررسی بریده‌شدن ورودی
        }
        analysis_results.append(item)
        print(f"بند {idx}/{len(cleaned_clauses)} ارزیابی شد | ریسک: {item['predicted_risk']} | طول: {item['char_count']} کاراکتر")
        
    except Exception as e:
        print(f"خطا در پردازش بند {idx}: {e}")
        analysis_results.append({
            "id": idx,
            "clause_text": clause,
            "error": str(e)
        })

# ۶. ذخیره خروجی کامل در فایل JSON
output_file = "contract_risk_detailed_analysis.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

print(f"\n✅ فایل '{output_file}' با موفقیت ذخیره شد. حالا کل متن هر بند و جزئیات ارزیابی داخلش قابل مشاهده است.")


Loading local checkpoint from: ./contract-risk-model
✅ مدل محلی با موفقیت لود شد.

تعداد 44 بند کامل برای ارزیابی استخراج شد.



Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 712.71it/s]


بند 1/44 ارزیابی شد | ریسک: کم‌ریسک | طول: 1382 کاراکتر
بند 2/44 ارزیابی شد | ریسک: مناسب | طول: 56 کاراکتر
بند 3/44 ارزیابی شد | ریسک: کم‌ریسک | طول: 298 کاراکتر
بند 4/44 ارزیابی شد | ریسک: مناسب | طول: 257 کاراکتر
بند 5/44 ارزیابی شد | ریسک: کم‌ریسک | طول: 231 کاراکتر
بند 6/44 ارزیابی شد | ریسک: کم‌ریسک | طول: 129 کاراکتر
بند 7/44 ارزیابی شد | ریسک: مناسب | طول: 185 کاراکتر
بند 8/44 ارزیابی شد | ریسک: کم‌ریسک | طول: 211 کاراکتر
بند 9/44 ارزیابی شد | ریسک: مناسب | طول: 245 کاراکتر
بند 10/44 ارزیابی شد | ریسک: مناسب | طول: 145 کاراکتر
بند 11/44 ارزیابی شد | ریسک: کم‌ریسک | طول: 244 کاراکتر
بند 12/44 ارزیابی شد | ریسک: مناسب | طول: 204 کاراکتر
بند 13/44 ارزیابی شد | ریسک: مناسب | طول: 168 کاراکتر
بند 14/44 ارزیابی شد | ریسک: مناسب | طول: 239 کاراکتر
بند 15/44 ارزیابی شد | ریسک: کم‌ریسک | طول: 112 کاراکتر
بند 16/44 ارزیابی شد | ریسک: مناسب | طول: 219 کاراکتر
بند 17/44 ارزیابی شد | ریسک: مناسب | طول: 219 کاراکتر
بند 18/44 ارزیابی شد | ریسک: مناسب | طول: 382 کاراکتر
بند 19/44 ارزیابی شد | 